In [24]:
import os
import fnmatch

def delete_non_filtered_tif_files(folder_path):
    # Deletes all files starting with 'Non-Filtered' and with the '.tif' extension
    # from the specified folder and its subfolders.
    if not os.path.isdir(folder_path):
        print(f"Error: The folder '{folder_path}' does not exist.")
        return

    # Traverse the folder and its subfolders
    for root, dirs, files in os.walk(folder_path):
        for filename in files:
            # Check if the file matches the naming criteria
            if fnmatch.fnmatch(filename, 'Non-Filtered*.tif'):
                file_path = os.path.join(root, filename)
                try:
                    os.remove(file_path)
                    print(f"Deleted: {file_path}")
                except Exception as e:
                    print(f"Error deleting file '{file_path}': {e}")

if __name__ == "__main__":
    
    target_folder = 'project_data' 
    delete_non_filtered_tif_files(target_folder)


Deleted: project_data/Two_phase_series/Two_phase_series_012/Non-Filtered_T_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_012/Non-Filtered_R_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_015/Non-Filtered_T_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_015/Non-Filtered_R_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_014/Non-Filtered_T_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_014/Non-Filtered_R_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_013/Non-Filtered_T_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_013/Non-Filtered_R_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_009/Non-Filtered_T_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_009/Non-Filtered_R_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_007/Non-Filtered_T_Image.tif
Deleted: project_data/Two_phase_series/Two_phase_series_007/Non-F

In [25]:
import os
import random
import shutil

# Define directory paths
random_series_path = './project_data/Random_series'
flux_series_path = './project_data/Flux_series'
time_series_path = './project_data/Time_series'
two_phase_series_path = './project_data/Two_phase_series'
boundary_series_path = './project_data/Boundary_series'

val_output_path = './project_data/validation_set'
test_output_path = './project_data/test_set'
train_output_path = './project_data/train_set'

# Define the split ratios
val_ratio = 1
test_ratio = 2
train_ratio = 7

# Create output directories if they do not exist
os.makedirs(val_output_path, exist_ok=True)
os.makedirs(test_output_path, exist_ok=True)
os.makedirs(train_output_path, exist_ok=True)

# Calculate total ratio
total_ratio = val_ratio + test_ratio + train_ratio

# Get list of second-level subfolders from Random_series
random_series_folders = [f.path for f in os.scandir(random_series_path) if f.is_dir()]

# Shuffle the list of subfolders randomly
random.shuffle(random_series_folders)

# Calculate the number of folders for each dataset
total_folders = len(random_series_folders)
val_count = int((val_ratio / total_ratio) * total_folders)
test_count = int((test_ratio / total_ratio) * total_folders)
train_count = total_folders - val_count - test_count  # The rest goes to training set

# Split the Random_series dataset into validation, test, and training sets
val_folders = random_series_folders[:val_count]
test_folders = random_series_folders[val_count:val_count + test_count]
train_folders = random_series_folders[val_count + test_count:]

# Function to get second-level subfolders from other series directories
def get_subfolders(series_path):
    return [f.path for f in os.scandir(series_path) if f.is_dir()]

# Add subfolders from Flux_series, Time_series, Two_phase_series, and Boundary_series to the training set
train_folders += get_subfolders(flux_series_path)
train_folders += get_subfolders(time_series_path)
train_folders += get_subfolders(two_phase_series_path)
train_folders += get_subfolders(boundary_series_path)

# Function to copy folders to the target output directory
def copy_folders(folders, output_path):
    for folder in folders:
        folder_name = os.path.basename(folder)
        dest_path = os.path.join(output_path, folder_name)
        # Check if the folder already exists, to avoid duplication
        if not os.path.exists(dest_path):
            shutil.copytree(folder, dest_path)
        else:
            print(f"Folder already exists, skipping: {dest_path}")

# Copy validation, test, and training folders to their respective directories
copy_folders(val_folders, val_output_path)
copy_folders(test_folders, test_output_path)
copy_folders(train_folders, train_output_path)

# Print the count of folders in each dataset
print(f"Validation set count: {len(val_folders)}, Test set count: {len(test_folders)}, Training set count: {len(train_folders)}")


Validation set count: 20, Test set count: 40, Training set count: 238


In [34]:
import os
import json
import pandas as pd

def classify_f(value, lower, upper):
    """
    Classify the value based on the provided boundaries.

    Parameters:
        value (float): The value from the 'F' column.
        lower (float): The lower boundary.
        upper (float): The upper boundary.

    Returns:
        int: The classification category (1, 2, or 3).
    """
    if value < lower:
        return 1
    elif lower <= value <= upper:
        return 2
    else:
        return 3

def process_set(set_name, project_data_dir='project_data'):
    set_dir = os.path.join(project_data_dir, set_name)
    
    # Check if the dataset directory exists
    if not os.path.isdir(set_dir):
        print(f"Warning: Directory {set_dir} does not exist. Skipping processing.")
        return pd.DataFrame()
    
    # Initialize an empty DataFrame to store results
    set_results = pd.DataFrame(columns=['x', 'y', 'Sn', 'Ba', 'Category'])
    
    # Iterate through all subfolders in the dataset directory
    for subfolder in os.listdir(set_dir):
        subfolder_path = os.path.join(set_dir, subfolder)
        
        if os.path.isdir(subfolder_path):
            settings_path = os.path.join(subfolder_path, 'settings.json')
            csv_path = os.path.join(subfolder_path, 'image_data.csv')
            
            # Check if settings.json and image_data.csv exist
            if not os.path.isfile(settings_path):
                print(f"Warning: {settings_path} does not exist. Skipping folder '{subfolder}'.")
                continue
            if not os.path.isfile(csv_path):
                print(f"Warning: {csv_path} does not exist. Skipping folder '{subfolder}'.")
                continue
            
            # Read settings.json and extract boundaries
            try:
                with open(settings_path, 'r', encoding='utf-8') as f:
                    settings = json.load(f)
            except Exception as e:
                print(f"Error: Unable to read {settings_path}. Error: {e}")
                continue
            
            boundaries = settings.get('boundaries', [])
            if len(boundaries) == 2:
                lower_bound = boundaries[0]
                upper_bound = boundaries[1]
                print(f"Processing folder: {subfolder} using boundaries: {lower_bound}, {upper_bound}")
            elif len(boundaries) == 4:
                lower_bound = boundaries[1]
                upper_bound = boundaries[2]
                print(f"Processing folder: {subfolder} using boundaries: {lower_bound}, {upper_bound}")
            else:
                print(f"Warning: 'boundaries' in {settings_path} has length {len(boundaries)} which is not supported. Skipping folder '{subfolder}'.")
                continue
            
            # Read image_data.csv file
            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print(f"Error: Unable to read {csv_path}. Error: {e}")
                continue
            
            # Check for necessary columns
            required_columns = ['x', 'y', 'Sn', 'Ba', 'F']
            missing_columns = [col for col in required_columns if col not in df.columns]
            if missing_columns:
                print(f"Warning: {csv_path} is missing columns: {missing_columns}. Skipping folder '{subfolder}'.")
                continue
            
            # Apply classification and add 'Category' column
            try:
                df['Category'] = df['F'].apply(lambda x: classify_f(x, lower_bound, upper_bound))
            except Exception as e:
                print(f"Error: Classification failed for folder '{subfolder}'. Error: {e}")
                continue
            
            # Select the required columns and append to the results DataFrame
            selected_columns = ['x', 'y', 'Sn', 'Ba', 'Category']
            set_results = pd.concat([set_results, df[selected_columns]], ignore_index=True)
    
    return set_results

def save_results(set_name, df, project_data_dir='project_data'):
    if df.empty:
        print(f"No data to save for {set_name}.")
        return
    
    # Determine the output filename based on the set name
    output_filename = ''
    if set_name == 'test_set':
        output_filename = 'test.csv'
    elif set_name == 'train_set':
        output_filename = 'train.csv'
    elif set_name == 'validation_set':
        output_filename = 'validation.csv'
    else:
        print(f"Unknown set name: {set_name}. No file will be saved.")
        return
    
    output_path = os.path.join(project_data_dir, output_filename)
    try:
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"Results for {set_name} have been saved to {output_path}")
    except Exception as e:
        print(f"Error: Unable to save results to {output_path}. Error: {e}")

def main(project_data_dir='project_data'):
    subsets = ['test_set', 'train_set', 'validation_set']
    
    for subset in subsets:
        print(f"\nStarting processing for subset: {subset}")
        results_df = process_set(subset, project_data_dir)
        save_results(subset, results_df, project_data_dir)

if __name__ == "__main__":
    main()



Starting processing for subset: test_set
Processing folder: Random_series_070 using boundaries: 0.65, 0.69
Processing folder: Random_series_084 using boundaries: 0.65, 0.69
Processing folder: Random_series_082 using boundaries: 0.6, 0.67
Processing folder: Random_series_013 using boundaries: 0.63, 0.68
Processing folder: Random_series_161 using boundaries: 0.6, 0.76
Processing folder: Random_series_103 using boundaries: 0.57, 0.76
Processing folder: Random_series_167 using boundaries: 0.55, 0.69
Processing folder: Random_series_160 using boundaries: 0.58, 0.7
Processing folder: Random_series_156 using boundaries: 0.59, 0.72
Processing folder: Random_series_105 using boundaries: 0.56, 0.68
Processing folder: Random_series_134 using boundaries: 0.6, 0.77
Processing folder: Random_series_133 using boundaries: 0.55, 0.67
Processing folder: Random_series_120 using boundaries: 0.58, 0.67
Processing folder: Random_series_187 using boundaries: 0.64, 0.73
Processing folder: Random_series_145 u

In [35]:
import pandas as pd
file_path = 'project_data/train.csv'
df = pd.read_csv(file_path)
first_100_rows = df.head(100)
print(first_100_rows)
print(df.shape)

       x     y            Sn            Ba  Category
0  -2.49 -0.49  3.062847e-07  2.233571e-07         2
1  -2.49 -0.44  3.053393e-07  2.225576e-07         2
2  -2.49 -0.39  3.043868e-07  2.217501e-07         2
3  -2.49 -0.34  3.034276e-07  2.209347e-07         2
4  -2.49 -0.29  3.024617e-07  2.201116e-07         2
..   ...   ...           ...           ...       ...
95 -2.34 -0.29  3.088415e-07  2.164520e-07         2
96 -2.34 -0.24  3.078350e-07  2.156442e-07         2
97 -2.34 -0.19  3.068219e-07  2.148293e-07         2
98 -2.34 -0.14  3.058024e-07  2.140074e-07         2
99 -2.34 -0.09  3.047767e-07  2.131786e-07         2

[100 rows x 5 columns]
(1925896, 5)


In [2]:
# Import necessary libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Load the training and testing datasets
train_path = 'project_data/train.csv'
test_path = 'project_data/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 2. Separate features and target variable
feature_cols = ['x', 'y', 'Sn', 'Ba']
target_col = 'Category'

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

# 3. Feature Scaling (important for Logistic Regression and KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Define the models to be used
models = {
    'Logistic Regression': LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

# 5. Train the models and evaluate accuracy
for name, model in models.items():
    # Use scaled features for models that require scaling
    if name in ['Logistic Regression', 'K-Nearest Neighbors']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy on Test Set: {accuracy:.4f}")


Logistic Regression Accuracy on Test Set: 0.8075
Decision Tree Accuracy on Test Set: 0.5580
K-Nearest Neighbors Accuracy on Test Set: 0.7538
